# Solutions — Bakehouse Customer Transactions (Medium 03)

**Datasets:** `sales_customers`, `sales_transactions`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

customers    = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")

## Solution 1 — Transaction Count & Spend per Country

In [ ]:
# Why: join on customerID then groupBy country
result_1 = (
    transactions
    .join(customers, on="customerID")
    .groupBy("country")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("totalPrice"), 2).alias("total_spend")
    )
    .orderBy(F.col("total_spend").desc())
)

## Solution 2 — Top 10 Customers by Total Spend

In [ ]:
result_2 = (
    transactions
    .join(customers, on="customerID")
    .groupBy("customerID","first_name","last_name","country")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_spend"))
    .orderBy(F.col("total_spend").desc())
    .limit(10)
)

## Solution 3 — Customers Who Never Transacted (Anti-Join)

In [ ]:
# Why: anti-join keeps left rows with NO match on the right
result_3 = (
    customers
    .join(transactions.select("customerID").distinct(), on="customerID", how="left_anti")
    .select("customerID","first_name","last_name","country")
)

## Solution 4 — Avg Basket Size per Country

In [ ]:
# Why: distinct products per transaction = basket size; avg that per country
basket = (
    transactions
    .join(customers, on="customerID")
    .groupBy("country","transactionID")
    .agg(F.countDistinct("product").alias("basket_size"))
)
result_4 = (
    basket
    .groupBy("country")
    .agg(F.round(F.avg("basket_size"), 2).alias("avg_basket_size"))
    .orderBy(F.col("avg_basket_size").desc())
)

## Solution 5 — First & Last Purchase per Customer

In [ ]:
result_5 = (
    transactions
    .join(customers, on="customerID")
    .groupBy("customerID","first_name","last_name")
    .agg(
        F.to_date(F.min("dateTime")).alias("first_purchase"),
        F.to_date(F.max("dateTime")).alias("last_purchase")
    )
    .orderBy("first_purchase")
)

## Solution 6 — Customers Who Bought the Same Product More Than Once

In [ ]:
result_6 = (
    transactions
    .groupBy("customerID","product")
    .agg(F.count("*").alias("purchase_count"))
    .filter(F.col("purchase_count") > 1)
    .orderBy(F.col("purchase_count").desc())
)

## Solution 7 — Revenue & Percentage per Continent

In [ ]:
# Why: global sum via window so % can be computed without a separate join
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_7 = (
    transactions
    .join(customers, on="customerID")
    .groupBy("continent")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
    .withColumn(
        "revenue_pct",
        F.round(F.col("total_revenue") / F.sum("total_revenue").over(w) * 100, 2)
    )
    .orderBy(F.col("revenue_pct").desc())
)